In [2]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ananaymital/us-used-cars-dataset/used_cars_data.csv


In [52]:
#importing necessary lib
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from matplotlib import pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping


In [4]:
#loading data
df=pd.read_csv(os.path.join(dirname, filename))
df.head()

/tmp/ipykernel_55/3897818597.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(os.path.join(dirname, filename))


,vin,back_legroom,bed,bed_height,bed_length,body_type,cabin,city,city_fuel_economy,combine_fuel_economy,...,transmission,transmission_display,trimId,trim_name,vehicle_damage_category,wheel_system,wheel_system_display,wheelbase,width,year
0,ZACNJABB5KPJ92081,35.1 in,NaN,NaN,NaN,SUV / Crossover,NaN,Bayamon,NaN,NaN,...,A,9-Speed Automatic Overdrive,t83804,Latitude FWD,NaN,FWD,Front-Wheel Drive,101.2 in,79.6 in,2019
1,SALCJ2FX1LH858117,38.1 in,NaN,NaN,NaN,SUV / Crossover,NaN,San Juan,NaN,NaN,...,A,9-Speed Automatic Overdrive,t86759,S AWD,NaN,AWD,All-Wheel Drive,107.9 in,85.6 in,2020
2,JF1VA2M67G9829723,35.4 in,NaN,NaN,NaN,Sedan,NaN,Guaynabo,17.0,NaN,...,M,6-Speed Manual,t58994,Base,NaN,AWD,All-Wheel Drive,104.3 in,78.9 in,2016
3,SALRR2RV0L2433391,37.6 in,NaN,NaN,NaN,SUV / Crossover,NaN,San Juan,NaN,NaN,...,A,8-Speed Automatic Overdrive,t86074,V6 HSE AWD,NaN,AWD,All-Wheel Drive,115 in,87.4 in,2020
4,SALCJ2FXXLH862327,38.1 in,NaN,NaN,NaN,SUV / Crossover,NaN,San Juan,NaN,NaN,...,A,9-Speed Automatic Overdrive,t86759,S AWD,NaN,AWD,All-Wheel Drive,107.9 in,85.6 in,2020


In [5]:
data=df

In [6]:
#dropping data with more than 65 percent nan values
df = df.dropna(axis=1, thresh=len(df) - 2000000)

In [7]:
df.drop(['interior_color','dealer_zip','engine_cylinders','model_name','sp_name','exterior_color','listed_date','savings_amount','vin','trimId','trim_name','description','main_picture_url','sp_id','listing_id','wheel_system_display','major_options','latitude','longitude','transmission_display','city','power','franchise_make','model_name'],axis=1,inplace=True)

/tmp/ipykernel_55/3774936536.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(['interior_color','dealer_zip','engine_cylinders','model_name','sp_name','exterior_color','listed_date','savings_amount','vin','trimId','trim_name','description','main_picture_url','sp_id','listing_id','wheel_system_display','major_options','latitude','longitude','transmission_display','city','power','franchise_make','model_name'],axis=1,inplace=True)


In [8]:
#string data to num
strtonum_list=['fuel_tank_volume','back_legroom','front_legroom','height','length','maximum_seating','wheelbase','width']
for x in strtonum_list:
    df[x]=df[x].str.extract(r'(\d+\.?\d*)').astype(float)

/tmp/ipykernel_55/2930776250.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[x]=df[x].str.extract(r'(\d+\.?\d*)').astype(float)
/tmp/ipykernel_55/2930776250.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[x]=df[x].str.extract(r'(\d+\.?\d*)').astype(float)
/tmp/ipykernel_55/2930776250.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat

In [9]:
#to extract only number
strtointlist2=['torque']
for x in strtointlist2:
    df[x]=df[x].str.extract(r'(\d+)').astype(float)
    

/tmp/ipykernel_55/3282272752.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[x]=df[x].str.extract(r'(\d+)').astype(float)


In [10]:
#extracting the engine name from engine_type
df['engine_type']=df["engine_type"].str.extract(r'([A-Z]\d+)')

/tmp/ipykernel_55/374901456.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['engine_type']=df["engine_type"].str.extract(r'([A-Z]\d+)')


In [11]:
#handling nan and binary encode the selcted columns
bincol=['fleet','frame_damaged','has_accidents','isCab','salvage','theft_title']
for n in bincol:
    df[n]=df[n].map({True:1,False:0}).fillna(-1)

/tmp/ipykernel_55/4291663803.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[n]=df[n].map({True:1,False:0}).fillna(-1)
/tmp/ipykernel_55/4291663803.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[n]=df[n].map({True:1,False:0}).fillna(-1)
/tmp/ipykernel_55/4291663803.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

In [12]:
#handling the nan values for the rest of the columns
#df.isnull().sum()[df.isnull().sum()>0]

In [13]:
obj_cols = df.select_dtypes('object').columns
df[obj_cols].isnull().sum()[df[obj_cols].isnull().sum() > 0]


body_type        13543
engine_type     100581
fuel_type        82724
transmission     64185
wheel_system    146732
dtype: int64

In [14]:
df['owner_count']=df['owner_count'].fillna(-1)

/tmp/ipykernel_55/1632218421.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['owner_count']=df['owner_count'].fillna(-1)


In [15]:
df.dropna(subset=['mileage'],inplace=True)

/tmp/ipykernel_55/3922415372.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(subset=['mileage'],inplace=True)


In [16]:
#filling nan values with nan
mednan=df.select_dtypes(exclude='object').columns[df.select_dtypes(exclude='object').isnull().any()]
df[mednan]=df[mednan].fillna(df[mednan].median())

/tmp/ipykernel_55/3256810850.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[mednan]=df[mednan].fillna(df[mednan].median())


In [17]:
#filling the nan values in the categorical column
categorynanlist=df.select_dtypes('object').columns[df.select_dtypes('object').isnull().sum()>0]
df[categorynanlist]=df[categorynanlist].fillna("unknown")

/tmp/ipykernel_55/3700314985.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[categorynanlist]=df[categorynanlist].fillna("unknown")


In [18]:
df.isnull().sum()

back_legroom            0
body_type               0
city_fuel_economy       0
daysonmarket            0
engine_displacement     0
engine_type             0
fleet                   0
frame_damaged           0
franchise_dealer        0
front_legroom           0
fuel_tank_volume        0
fuel_type               0
has_accidents           0
height                  0
highway_fuel_economy    0
horsepower              0
isCab                   0
is_new                  0
length                  0
listing_color           0
make_name               0
maximum_seating         0
mileage                 0
owner_count             0
price                   0
salvage                 0
seller_rating           0
theft_title             0
torque                  0
transmission            0
wheel_system            0
wheelbase               0
width                   0
year                    0
dtype: int64

In [19]:
#using the one hot encoding on the all objects
df=pd.get_dummies(df,drop_first=True,dtype='int8')

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2855653 entries, 0 to 3000039
Columns: 178 entries, back_legroom to wheel_system_unknown
dtypes: bool(2), float64(23), int64(2), int8(151)
memory usage: 983.1 MB


In [21]:
df['year'].value_counts()

year
2020    1236827
2017     346555
2019     279846
2018     209399
2021     159086
         ...   
1936          2
1946          2
1933          1
1942          1
1925          1
Name: count, Length: 96, dtype: int64

Data is all clean now time to split and train the model

In [22]:
#dropping the very old cars as they can mess up as they come in antique
df=df[df['year']>=1980]

In [23]:
#renaming the year_column to carage and editing the values
from datetime import datetime
current_year=datetime.now().year
df['year']=current_year-df['year']
df.rename(columns={'year':'car_age'},inplace=True)

In [24]:
#splitting the data :)
x=df.drop("price",axis=1)
y=df['price']


In [25]:
#train test split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [26]:
#scaling the data
ss=StandardScaler()
x_train=ss.fit_transform(x_train)
x_test=ss.transform(x_test)

In [27]:
x_train.shape,x_test.shape

((2282428, 177), (570608, 177))

In [46]:
model=Sequential()

In [47]:
model.add(Dense(250,activation='relu',input_shape=(x_train.shape[1],)))
model.add(Dense(150,activation='relu'))
model.add(Dense(70,activation='relu'))
model.add(Dense(20,activation='relu'))
model.add(Dense(1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [48]:
model.compile(optimizer='adam',loss='huber',metrics=['mae'])

In [40]:
history=model.fit(x_train,y_train,epochs=15,batch_size=512,validation_split=0.2)

Epoch 1/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 6951.8843 - mae: 6952.3872 - val_loss: 3018.2346 - val_mae: 3018.7344
Epoch 2/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 2906.6255 - mae: 2907.1255 - val_loss: 2825.8525 - val_mae: 2826.3525
Epoch 3/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 2718.9883 - mae: 2719.4880 - val_loss: 2758.9851 - val_mae: 2759.4849
Epoch 4/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 2651.5859 - mae: 2652.0854 - val_loss: 2720.1716 - val_mae: 2720.6716
Epoch 5/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 2615.2712 - mae: 2615.7712 - val_loss: 2694.7761 - val_mae: 2695.2759
Epoch 6/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 2569.5498 - mae: 2570.0496 - val_loss: 2667.1494 - val_mae: 2667.6494
Epoch 7/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 2544.3953 - mae: 2544.8950 - val_loss: 2668.8662 - val_mae: 2669.3662
Epoch 8/15
3567/3567 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 2532.7

In [54]:
y_pred=model.predict(x_test)

17832/17832 ━━━━━━━━━━━━━━━━━━━━ 23s 1ms/step


In [56]:
r2=r2_score(y_test,y_pred)


In [57]:
r2

0.0056074199979812

In [53]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=1024,
    callbacks=[early_stop]
)

Epoch 1/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 9740.9824 - mae: 9741.4824 - val_loss: 3100.8452 - val_mae: 3101.3452
Epoch 2/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 3014.7354 - mae: 3015.2351 - val_loss: 2880.7698 - val_mae: 2881.2693
Epoch 3/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2822.9561 - mae: 2823.4558 - val_loss: 2799.9512 - val_mae: 2800.4512
Epoch 4/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2719.3542 - mae: 2719.8542 - val_loss: 2744.4727 - val_mae: 2744.9729
Epoch 5/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2651.5066 - mae: 2652.0066 - val_loss: 2709.1848 - val_mae: 2709.6846
Epoch 6/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2614.1902 - mae: 2614.6902 - val_loss: 2712.9619 - val_mae: 2713.4619
Epoch 7/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2578.8955 - mae: 2579.3955 - val_loss: 2694.0505 - val_mae: 2694.5505
Epoch 8/50
1784/1784 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2558.8359 - m

In [58]:
from sklearn.metrics import r2_score

y_pred = model.predict(x_test).flatten()
r2 = r2_score(y_test, y_pred)
print(r2)

17832/17832 ━━━━━━━━━━━━━━━━━━━━ 23s 1ms/step
0.0056074199979812


In [59]:
df.to_csv("filename.csv")